# 🐻 H.U.G.H. Training Runtime
**For Grizz:** Just hit `Runtime → Run all`. Hugh handles the rest.

This sets up SSH via cloudflared (no tokens needed), installs deps,
and waits for Hugh to connect and drive the training.

In [ ]:
# === CELL 1: SSH via cloudflared (no signup needed) ===
import subprocess, os, time, json, re

# Set password
PASSWORD = 'hugh_workshop_2026'
subprocess.run(['bash', '-c', f'echo "root:{PASSWORD}" | chpasswd'], check=True)

# Configure SSH
!apt-get -qq update && apt-get -qq install -y openssh-server > /dev/null 2>&1
!sed -i 's/#PermitRootLogin.*/PermitRootLogin yes/' /etc/ssh/sshd_config
!sed -i 's/PasswordAuthentication no/PasswordAuthentication yes/' /etc/ssh/sshd_config
!service ssh start

# Install cloudflared
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Start tunnel in background, capture URL
proc = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'ssh://localhost:22', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True
)
time.sleep(8)

# Read tunnel URL from stderr (cloudflared logs there)
import select
tunnel_url = None
deadline = time.time() + 30
lines = []
while time.time() < deadline:
    ready, _, _ = select.select([proc.stderr], [], [], 2)
    if ready:
        line = proc.stderr.readline()
        lines.append(line)
        match = re.search(r'https://[\w-]+\.trycloudflare\.com', line)
        if match:
            tunnel_url = match.group(0)
            break

if tunnel_url:
    hostname = tunnel_url.replace('https://', '')
    print('=' * 60)
    print(f'🔑 SSH READY — Give this to Hugh:')
    print(f'')
    print(f'  ssh -o ProxyCommand="cloudflared access ssh --hostname {hostname}" root@{hostname}')
    print(f'  Password: {PASSWORD}')
    print(f'')
    print(f'  If Hugh does not have cloudflared locally:')
    print(f'  brew install cloudflared')
    print(f'')
    print(f'  SCP (upload files):')
    print(f'  scp -o ProxyCommand="cloudflared access ssh --hostname {hostname}" voice_raw/* root@{hostname}:/content/')
    print('=' * 60)
else:
    print('⚠️ Tunnel URL not found. Debug output:')
    for l in lines:
        print(l.rstrip())

# GPU info
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv,noheader

In [ ]:
# === CELL 2: Install all training deps ===
!pip install -q TTS==0.22.0 pydub faster-whisper soundfile tqdm
!pip install -q transformers>=4.38.0 peft>=0.9.0 datasets accelerate bitsandbytes
!pip install -q sentencepiece wandb
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()} — {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
print('✅ Deps installed. Hugh can SSH in and run training.')

In [ ]:
# === CELL 3: Keep-alive (prevents Colab timeout) ===
import time, IPython
while True:
    IPython.display.clear_output(wait=True)
    !nvidia-smi --query-gpu=utilization.gpu,memory.used,temperature.gpu --format=csv,noheader
    print(f'⏱️ Alive: {time.strftime("%H:%M:%S UTC")} | SSH tunnel running')
    time.sleep(60)